# P046: 広告ログの異常な値を探す

Level 3 ・ 探索(EDA) ・ 目安 10 分

**解答と解説**

In [ ]:
import pandas as pd

df = pd.read_csv("data/ad_logs.csv")

## 1. 推奨確認項目

1. 期間と行数: `date` の min / max / nunique、広告数 × 日数と行数を比べる
2. 指標間の矛盾: `clicks > impressions`、`conversions > clicks`
3. ありえない値: `describe()` の min / max、負の値
4. 重複: `duplicated().sum()`
5. カテゴリの表記ゆれ: `value_counts()`
6. 異常値を除く前後で集計を比べる

## 2. 模範的なコード

In [ ]:
df.shape, df["date"].min(), df["date"].max(), df["date"].nunique(), df["ad_id"].nunique()

In [ ]:
df.describe()

In [ ]:
df[df["clicks"] > df["impressions"]]

In [ ]:
df[df["cost"] < 0]

In [ ]:
df[df["impressions"] == 0]

In [ ]:
df.duplicated().sum()

In [ ]:
df["campaign"].value_counts()

In [ ]:
# 除外前の CTR
raw = df.groupby("campaign")[["clicks", "impressions"]].sum()
raw["clicks"] / raw["impressions"]

In [ ]:
# 重複・矛盾・負の値を除き、表記ゆれをそろえてから CTR
clean = df.drop_duplicates()
clean = clean[(clean["clicks"] <= clean["impressions"]) & (clean["cost"] >= 0)]
clean["campaign"] = clean["campaign"].str.lower()
agg = clean.groupby("campaign")[["clicks", "impressions"]].sum()
agg["clicks"] / agg["impressions"]

## 3. このデータで観察すべきポイント

1. 期間は 2025-05-01 〜 05-31(31日)、広告は8種類。想定は 248 行だが **251 行** ある → **完全重複が3行**
2. **clicks > impressions の行が2行**(ad 10006 の 05-05、ad 10007 の 05-19)。clicks の max が 5346 と describe でも目立つ
3. **cost が負(-4200)の行が1行**(ad 10002 の 05-26)
4. impressions が 0 の日が1行(ad 10007 の 05-15)。配信停止日と思われる。行ごとに CTR を計算すると 0/0 で NaN になる
5. `campaign` に **`"Brand"` と `"brand"` の表記ゆれ**(4行)
6. **retarget の CTR は、除外前 約6.3% → 除外後 約3.5%**。異常な2行だけで CTR がほぼ倍に見えていた。
   spring_sale(約2.0%)、brand(約0.8%)はほぼ変わらない

## 4. 今回覚えるポイント

- 「ありえない値」は**指標の定義から考える**: clicks ≤ impressions、cost ≥ 0 など
- 見つけたら `df[条件]` で該当行を表示して中身を見る
- **想定行数(日数 × 対象数)と実際の行数を比べる**と、重複や欠けに気づける
- 少数の異常値が集計結果を大きく変えることがある。除外前後で比べる